# Feature Engineering - 보험 청구 사기 탐지
## 2025 NESS Statathon

**목표**: 행동·리스크 기반의 예측 피처를 생성하여 사기 탐지 모델의 성능을 극대화

**피처 설계 원칙**:
- 사기 조사관의 시각으로 접근 (탐문·패턴 기반)
- 타겟 누출(data leakage) 완전 방지
- ID 변수(`claim_number`, `zip_code` 직접 사용) 배제
- 비율(ratio), 빈도(frequency), 불안정성(instability), 의심 패턴 지표 위주 생성

---
### 피처 카테고리 개요
| 카테고리 | 설명 |
|---|---|
| A. 운전자 리스크 | 나이 이상치, 안전등급 역수, 소득 불일치 |
| B. 청구 행동 | 과거 청구 빈도, 목격자 부재, 고책임 비율 |
| C. 차량 이상 | 가격-무게 불일치, 연식 대비 고액 청구 |
| D. 상호작용 피처 | 운전자×차량×청구 복합 위험 신호 |
| E. 날짜·시간 | 주말 청구, 월말 패턴, 계절성 |
| F. 지역 집계 | zip_code 핫스팟 (누출 방지 처리) |
| G. 종합 리스크 스코어 | 개별 위험 지표의 누적 점수 |

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import StratifiedKFold

DATA_DIR = Path('.')
train = pd.read_csv(DATA_DIR / 'train_2025.csv')
test  = pd.read_csv(DATA_DIR / 'test_2025.csv')

print(f"Train: {train.shape}  |  Test: {test.shape}")
print(f"Fraud rate: {train['fraud'].mean():.2%}")

Train: (18000, 25)  |  Test: (12000, 24)
Fraud rate: 15.82%


---
## 0. 전처리 유틸리티
피처 생성 전 Train+Test를 합쳐 일관된 변환을 적용하고, 마지막에 다시 분리

In [2]:
# Train / Test 합치기 (타겟 컬럼 제외)
train['_is_train'] = 1
test['_is_train']  = 0

# test에는 fraud 컬럼 없음 → 임시 NaN 추가
if 'fraud' not in test.columns:
    test['fraud'] = np.nan

df = pd.concat([train, test], axis=0, ignore_index=True)
print(f"Combined shape: {df.shape}")

# claim_date 파싱
df['claim_date'] = pd.to_datetime(df['claim_date'], format='%m/%d/%Y', errors='coerce')

Combined shape: (30000, 26)


---
## A. 운전자 리스크 피처 (Driver Risk Features)

**사기 조사관 관점**: 비정상적인 나이(데이터 조작 가능성), 낮은 안전등급, 소득 대비 고가 차량 소유는 사기 위험 신호

In [3]:
# A1. 나이 이상치 플래그
# age_of_driver 최대값이 249 → 신원 위조나 데이터 조작 의심
# 18 ~ 75세를 벗어나면 이상 신호
df['age_is_outlier'] = ((df['age_of_driver'] < 18) | (df['age_of_driver'] > 75)).astype(int)

# A2. 고위험 연령대 플래그 (청년·고령 운전자는 사고 확률 높음)
# 사기는 보상 금액이 크게 나오기 쉬운 고위험 profile에서 자주 발생
df['is_young_driver']  = (df['age_of_driver'] < 25).astype(int)
df['is_senior_driver'] = (df['age_of_driver'] > 65).astype(int)

# A3. 안전등급 역수 (낮을수록 위험 → 사기 위험도 비례)
# safty_rating이 낮은 운전자가 사기 청구를 낼 유인이 더 큼
df['safety_risk_score'] = 100 - df['safty_rating']  # 0=안전, 99=위험

# A4. 안전등급 매우 낮음 플래그 (하위 20%)
safety_q20 = df.loc[df['_is_train'] == 1, 'safty_rating'].quantile(0.20)
df['low_safety_flag'] = (df['safty_rating'] < safety_q20).astype(int)

# A5. 소득 구간 (절대적 소득보다 상대 위치가 중요)
# 저소득층은 보험금 필요성이 높아 사기 유인 증가
income_q25 = df.loc[df['_is_train'] == 1, 'annual_income'].quantile(0.25)
income_q75 = df.loc[df['_is_train'] == 1, 'annual_income'].quantile(0.75)
df['is_low_income']  = (df['annual_income'] < income_q25).astype(int)
df['is_high_income'] = (df['annual_income'] > income_q75).astype(int)

# A6. 소득 로그 변환 (스케일 정규화 + 극단값 완화)
df['log_annual_income'] = np.log1p(df['annual_income'])

In [4]:
summary_A = []

driver_risk_features = [
    'age_is_outlier',
    'is_young_driver',
    'is_senior_driver',
    'low_safety_flag',
    'is_low_income',
    'is_high_income'
]

for col in driver_risk_features:
    summary_A.append({
        'feature': col,
        'train_cnt': df.loc[df['_is_train']==1, col].sum() if col in df.columns else None,
        'test_cnt': df.loc[df['_is_train']==0, col].sum() if col in df.columns else None,
        'train_rate': df.loc[df['_is_train']==1, col].mean() if col in df.columns else None,
        'test_rate': df.loc[df['_is_train']==0, col].mean() if col in df.columns else None
    })

summary_A_df = pd.DataFrame(summary_A)
print("\n[A] Driver Risk Feature Summary")
display(summary_A_df)


[A] Driver Risk Feature Summary


,feature,train_cnt,test_cnt,train_rate,test_rate
0,age_is_outlier,190,115,0.010556,0.009583
1,is_young_driver,585,425,0.032500,0.035417
2,is_senior_driver,802,477,0.044556,0.039750
3,low_safety_flag,3487,2255,0.193722,0.187917
4,is_low_income,4500,2903,0.250000,0.241917
5,is_high_income,4500,3004,0.250000,0.250333


---
## B. 청구 행동 피처 (Claim Behavior Features)

**사기 조사관 관점**: 반복 청구자, 목격자 없음, 100% 책임 인정, 경찰 미신고는 사기의 고전적 패턴

In [5]:
# B1. 과거 청구 빈도 피처
# 반복 청구자(serial claimer)는 사기 가능성이 통계적으로 유의미하게 높음
df['is_repeat_claimer'] = (df['past_num_of_claims'] > 0).astype(int)
df['is_serial_claimer'] = (df['past_num_of_claims'] >= 5).astype(int)
df['is_extreme_claimer'] = (df['past_num_of_claims'] >= 15).astype(int)
df['log_past_claims'] = np.log1p(df['past_num_of_claims'])

# B2. 목격자 부재 (no witness = 검증 불가 = 고위험)
# 목격자가 없을 때 사기를 탐지하기 매우 어려움
# NaN 처리: 목격자 정보 자체가 없으면 '미확인'으로 별도 플래그
df['witness_absent'] = ((df['witness_present_ind'] == 0)).astype(int)
df['witness_unknown'] = df['witness_present_ind'].isna().astype(int)
# 목격자 없음 또는 미확인 = 강한 위험 신호
df['no_verification'] = ((df['witness_absent'] == 1) | (df['witness_unknown'] == 1)).astype(int)

# B3. 책임 비율 이상 패턴
# liab_prct = 100은 '상대방이 100% 내 잘못' 주장 → 과도한 보상 청구 의심
# liab_prct = 0은 '내 잘못 없음' 주장인데 청구함 → 사기 또는 무리한 청구
df['full_liability_flag'] = (df['liab_prct'] == 100).astype(int)
df['zero_liability_flag'] = (df['liab_prct'] == 0).astype(int)
df['extreme_liability_flag'] = ((df['liab_prct'] >= 90) | (df['liab_prct'] == 0)).astype(int)

# B4. 경찰 미신고 (policy_report_filed_ind = 0)
# 진짜 사고라면 보통 경찰에 신고 → 미신고는 경미하거나 조작 가능성
df['no_police_report'] = (df['policy_report_filed_ind'] == 0).astype(int)

# B5. 청구 예상금액 이상 패턴
# 0원 청구 예상 + 청구 제기 = 모순적 패턴
df['zero_payout_claim'] = (df['claim_est_payout'] == 0).astype(int)
df['log_claim_payout'] = np.log1p(df['claim_est_payout'])

# B6. 청구액 구간별 플래그 (Train 분위수 기준)
payout_q75 = df.loc[df['_is_train'] == 1, 'claim_est_payout'].quantile(0.75)
payout_q95 = df.loc[df['_is_train'] == 1, 'claim_est_payout'].quantile(0.95)
df['high_payout_flag'] = (df['claim_est_payout'] > payout_q75).astype(int)
df['extreme_payout_flag'] = (df['claim_est_payout'] > payout_q95).astype(int)

# B7. 사고 현장 유형
# 주차장 사고는 조작이 상대적으로 쉬움
df['accident_parking'] = (df['accident_site'] == 'Parking Lot').astype(int)
df['accident_highway'] = (df['accident_site'] == 'Highway').astype(int)
df['accident_local'] = (df['accident_site'] == 'Local').astype(int)

In [6]:
claim_behavior_features = [
    'is_repeat_claimer',
    'is_serial_claimer',
    'is_extreme_claimer',
    'witness_absent',
    'witness_unknown',
    'no_verification',
    'full_liability_flag',
    'zero_liability_flag',
    'extreme_liability_flag',
    'no_police_report',
    'zero_payout_claim',
    'high_payout_flag',
    'extreme_payout_flag',
    'accident_parking',
    'accident_highway',
    'accident_local'
]

summary_B = []

for col in claim_behavior_features:
    summary_B.append({
        'feature': col,
        'train_cnt': df.loc[df['_is_train']==1, col].sum() if col in df.columns else None,
        'test_cnt': df.loc[df['_is_train']==0, col].sum() if col in df.columns else None,
        'train_rate': df.loc[df['_is_train']==1, col].mean() if col in df.columns else None,
        'test_rate': df.loc[df['_is_train']==0, col].mean() if col in df.columns else None
    })

summary_B_df = pd.DataFrame(summary_B)

print("\n[B] Claim Behavior Feature Summary")
display(summary_B_df)


[B] Claim Behavior Feature Summary


,feature,train_cnt,test_cnt,train_rate,test_rate
0,is_repeat_claimer,8098,5533,0.449889,0.461083
1,is_serial_claimer,5366,3745,0.298111,0.312083
2,is_extreme_claimer,1134,861,0.063000,0.071750
3,witness_absent,13631,9148,0.757278,0.762333
4,witness_unknown,132,88,0.007333,0.007333
5,no_verification,13763,9236,0.764611,0.769667
6,full_liability_flag,740,477,0.041111,0.039750
7,zero_liability_flag,748,523,0.041556,0.043583
8,extreme_liability_flag,4127,2697,0.229278,0.224750
9,no_police_report,7136,4839,0.396444,0.403250


---
## C. 차량 이상 피처 (Vehicle Anomaly Features)

**사기 조사관 관점**: 차량 연식 대비 너무 높은 가격, 가격-무게 비율 불일치, 소득 대비 고가 차량은 차량 가치 부풀리기 사기의 핵심 신호

In [7]:
# C1. 차량 가격 관련 피처
df['log_vehicle_price']  = np.log1p(df['vehicle_price'])
df['log_vehicle_weight'] = np.log1p(df['vehicle_weight'])

# C2. 차량 가격 대비 무게 비율
# 정상 차량: 고가일수록 보통 무거움
# 이상치: 매우 가볍지만 매우 비싸거나, 무겁지만 매우 저렴
# 1e-6 : division by zero 방지용
df['price_per_weight'] = df['vehicle_price'] / (df['vehicle_weight'] + 1e-6)

# C3. 차량 연식에 따른 잔존 가치 추정
# 연식이 오래될수록 차량 가치는 하락 → 청구액이 잔존가치를 초과하면 의심
# 단순 감가 상각 : 매년 10% 하락
DEPRECIATION_RATE = 0.10
df['residual_value_ratio'] = np.maximum(1 - DEPRECIATION_RATE * (df['age_of_vehicle'] - 1), 0.05)
df['estimated_residual_value'] = df['vehicle_price'] * df['residual_value_ratio']

# C4. 청구 예상액 대비 차량 잔존가치 비율
# claim_est_payout / estimated_residual_value > 1 → 차량 가치 초과 청구
df['payout_to_residual_ratio'] = (
    df['claim_est_payout'] / (df['estimated_residual_value'] + 1e-6)
)
df['over_value_claim_flag'] = (df['payout_to_residual_ratio'] > 1.0).astype(int)

# C5. 청구 예상액 대비 차량 원가 비율
# 원가의 50% 이상을 청구하는 경우 과도한 수리비 청구 의심
HIGH_PAYOUT_RATIO_TH = 0.5
df['payout_to_price_ratio'] = (
    df['claim_est_payout'] / (df['vehicle_price'] + 1e-6)
)
df['high_payout_ratio_flag'] = (df['payout_to_price_ratio'] > HIGH_PAYOUT_RATIO_TH).astype(int)

# C6. 구형 차량 + 고가 청구 조합
# 오래된 차에 비해 청구액이 크면 → 수리비 부풀리기 의심
old_vehicle_threshold = df.loc[df['_is_train'] == 1, 'age_of_vehicle'].quantile(0.75)
df['old_vehicle_flag'] = (df['age_of_vehicle'] > old_vehicle_threshold).astype(int)
df['old_vehicle_high_claim'] = (
    (df['old_vehicle_flag'] == 1) & (df['high_payout_flag'] == 1)
).astype(int)

# C7. 신형 차량 + 즉시 청구 패턴 (구매 직후 사고 신고)
# 차량 연식 1년 = 구매 직후 청구 → 의도적 사고 유발 의심
df['new_vehicle_flag'] = (df['age_of_vehicle'] == 1).astype(int)
df['new_vehicle_high_claim'] = (
    (df['new_vehicle_flag'] == 1) & (df['high_payout_flag'] == 1)
).astype(int)

# C8. 차량 카테고리 인코딩 (Large > Medium > Compact)
vehicle_ohe = pd.get_dummies(
    df['vehicle_category'],
    prefix='veh_size',
    drop_first=True, 
    dummy_na=True
).astype(int)
df = pd.concat([df, vehicle_ohe], axis=1)
df.drop(columns=['vehicle_category'], inplace=True)

In [8]:
vehicle_anomaly_features = [
    'price_per_weight',
    'payout_to_residual_ratio',
    'over_value_claim_flag',
    'payout_to_price_ratio',
    'high_payout_ratio_flag',
    'old_vehicle_flag',
    'old_vehicle_high_claim',
    'new_vehicle_flag',
    'new_vehicle_high_claim'
]

# one-hot도 자동 포함 (veh_size prefix)
vehicle_anomaly_features += [c for c in df.columns if c.startswith('veh_size_')]

summary_C = []

for col in vehicle_anomaly_features:

    if df[col].nunique() <= 10:
        # binary / categorical feature → count, rate
        summary_C.append({
            'feature': col,
            'train_cnt': df.loc[df['_is_train']==1, col].sum() if df[col].dtype != 'float' else np.nan,
            'test_cnt': df.loc[df['_is_train']==0, col].sum() if df[col].dtype != 'float' else np.nan,
            'train_rate': df.loc[df['_is_train']==1, col].mean(),
            'test_rate': df.loc[df['_is_train']==0, col].mean(),
            'train_mean': df.loc[df['_is_train']==1, col].mean(),
            'test_mean': df.loc[df['_is_train']==0, col].mean()
        })
    else:
        # continuous feature → mean / std
        summary_C.append({
            'feature': col,
            'train_cnt': np.nan,
            'test_cnt': np.nan,
            'train_rate': np.nan,
            'test_rate': np.nan,
            'train_mean': df.loc[df['_is_train']==1, col].mean(),
            'test_mean': df.loc[df['_is_train']==0, col].mean()
        })

summary_C_df = pd.DataFrame(summary_C)

print("\n[C] Vehicle Anomaly Feature Summary")
display(summary_C_df)


[C] Vehicle Anomaly Feature Summary


,feature,train_cnt,test_cnt,train_rate,test_rate,train_mean,test_mean
0,price_per_weight,NaN,NaN,NaN,NaN,1.794655,1.803280
1,payout_to_residual_ratio,NaN,NaN,NaN,NaN,0.235631,0.227190
2,over_value_claim_flag,693.0,454.0,0.038500,0.037833,0.038500,0.037833
3,payout_to_price_ratio,NaN,NaN,NaN,NaN,0.091305,0.089741
4,high_payout_ratio_flag,613.0,391.0,0.034056,0.032583,0.034056,0.032583
5,old_vehicle_flag,4198.0,2784.0,0.233222,0.232000,0.233222,0.232000
6,old_vehicle_high_claim,1052.0,735.0,0.058444,0.061250,0.058444,0.061250
7,new_vehicle_flag,11058.0,7411.0,0.614333,0.617583,0.614333,0.617583
8,new_vehicle_high_claim,2768.0,1821.0,0.153778,0.151750,0.153778,0.151750
9,veh_size_Large,6021.0,3938.0,0.334500,0.328167,0.334500,0.328167


---
## D. 상호작용 피처 (Interaction Features)

**사기 조사관 관점**: 단일 위험 신호보다 복합 위험 신호의 조합이 훨씬 강력한 사기 예측 변수

In [9]:
# D1. 소득 대비 차량 가격 비율
# 연소득의 몇 배 차량인가? 정상 범위(0.3~1.5배)를 크게 벗어나면 의심
# 극단적으로 비싼 차: 대출 사기 또는 차량 가치 부풀리기
df['vehicle_to_income_ratio'] = df['vehicle_price'] / (df['annual_income'] + 1e-6)
# 소득 대비 차량이 너무 비쌈 플래그 (소득의 2배 이상)
ratio_q90 = df.loc[df['_is_train'] == 1, 'vehicle_to_income_ratio'].quantile(0.90)
df['overpriced_vehicle_flag'] = (
    df['vehicle_to_income_ratio'] > ratio_q90
).astype(int)

# D2. 청구액 대비 소득 비율
# 연소득 대비 청구금액이 클수록 사기 유인 높음
df['payout_to_income_ratio'] = df['claim_est_payout'] / (df['annual_income'] + 1e-6)

# D3. 저소득 + 소득 대비 고가 차량 + 고액 청구 = 전형적 사기 조합
df['low_income_overpriced_vehicle'] = (
    (df['is_low_income'] == 1) & (df['overpriced_vehicle_flag'] == 1)
).astype(int)
df['low_income_high_claim'] = (
    (df['is_low_income'] == 1) & (df['high_payout_flag'] == 1)
).astype(int)

# D4. 목격자 없음 + 경찰 미신고 조합
# 두 검증 수단이 모두 없음 → 가장 강력한 미검증 사기 신호
df['no_witness_no_report'] = (
    (df['no_verification'] == 1) & (df['no_police_report'] == 1)
).astype(int)

# D5. 반복 청구자 + 목격자 없음 조합
# 과거에도 청구했는데 이번에도 증인이 없음 → 상습 무증인 청구
df['repeat_no_witness'] = (
    (df['is_repeat_claimer'] == 1) & (df['no_verification'] == 1)
).astype(int)

# D6. 주소 이전 + 임차인 + 과거 청구 조합
# 불안정한 생활 패턴(이사, 임차)에 반복 청구 → 의도적 청구 패턴
df['living_status_rent'] = (df['living_status'] == 'Rent').astype(int)
df['unstable_lifestyle'] = (
    (df['address_change_ind'] == 1) &
    (df['living_status_rent'] == 1) &
    (df['is_repeat_claimer'] == 1)
).astype(int)

# D7. 젊은 운전자 + 고가 차량
# 25세 미만이 고가 차량 = 비현실적 → 차량 가치 부풀리기 또는 명의 대여
price_q75 = df.loc[df['_is_train']==1, 'vehicle_price'].quantile(0.75)
df['expensive_vehicle_flag'] = (df['vehicle_price'] > price_q75).astype(int)
df['young_expensive_vehicle'] = (
    (df['is_young_driver'] == 1) & (df['expensive_vehicle_flag'] == 1)
).astype(int)

# D8. 브로커 채널 + 경찰 미신고
# 브로커를 통한 청구 + 경찰 신고 없음 = 중간 공모 사기 의심
df['channel_broker'] = (df['channel'] == 'Broker').astype(int)
df['channel_online']  = (df['channel'] == 'Online').astype(int)
df['broker_no_report'] = (
    (df['channel_broker'] == 1) & (df['no_police_report'] == 1)
).astype(int)

# D9. 운전 위험도 높음 + 100% 책임 주장
# 운전 위험도 높은 운전자가 상대에게 100% 책임 전가 → 무리한 청구 가능성
df['unsafe_full_liability'] = (
    (df['low_safety_flag'] == 1) & (df['full_liability_flag'] == 1)
).astype(int)

# D10. 소득 대비 안전등급 (고소득-저안전 불일치)
# 금전적 여유가 있는데도 안전 관리 소홀 → 의도적 패턴 가능성
df['income_safety_interaction'] = df['annual_income'] * df['safety_risk_score']

In [10]:
interaction_features = [
    'overpriced_vehicle_flag',
    'low_income_overpriced_vehicle',
    'low_income_high_claim',
    'no_witness_no_report',
    'repeat_no_witness',
    'living_status_rent',
    'unstable_lifestyle',
    'expensive_vehicle_flag',
    'young_expensive_vehicle',
    'channel_broker',
    'channel_online',
    'broker_no_report',
    'unsafe_full_liability'
]

summary_D = []

for col in interaction_features:
    summary_D.append({
        'feature': col,
        'train_cnt': df.loc[df['_is_train']==1, col].sum() if col in df.columns else None,
        'test_cnt': df.loc[df['_is_train']==0, col].sum() if col in df.columns else None,
        'train_rate': df.loc[df['_is_train']==1, col].mean() if col in df.columns else None,
        'test_rate': df.loc[df['_is_train']==0, col].mean() if col in df.columns else None
    })
summary_D_df = pd.DataFrame(summary_D)

print("\n[D] Interaction Feature Summary")
display(summary_D_df)


[D] Interaction Feature Summary


,feature,train_cnt,test_cnt,train_rate,test_rate
0,overpriced_vehicle_flag,1800,1111,0.100000,0.092583
1,low_income_overpriced_vehicle,1266,743,0.070333,0.061917
2,low_income_high_claim,1121,741,0.062278,0.061750
3,no_witness_no_report,5670,3879,0.315000,0.323250
4,repeat_no_witness,6194,4291,0.344111,0.357583
5,living_status_rent,8002,5390,0.444556,0.449167
6,unstable_lifestyle,2032,1489,0.112889,0.124083
7,expensive_vehicle_flag,4500,3057,0.250000,0.254750
8,young_expensive_vehicle,149,116,0.008278,0.009667
9,channel_broker,9616,6387,0.534222,0.532250


---
## E. 날짜·시간 기반 피처 (Temporal Features)

**사기 조사관 관점**: 주말·공휴일 사고는 목격자가 적고 검증이 어렵습니다. 특히 월말(연말)에 청구가 집중되면 의심 가능

In [11]:
# E1. 날짜 기본 분해
df['claim_month']   = df['claim_date'].dt.month
df['claim_year']    = df['claim_date'].dt.year
df['claim_quarter'] = df['claim_date'].dt.quarter
df['claim_day']     = df['claim_date'].dt.day

# E2. 주말 청구 플래그
# 주말은 경찰·병원 배치가 적어 사고 조작에 유리
weekend_days = ['Saturday', 'Sunday']
df['is_weekend_claim'] = df['claim_day_of_week'].isin(weekend_days).astype(int)

# E3. 월요일 청구 플래그
# '주말에 사고 후 월요일에 신고' 패턴 → 시간 지연 신고 = 증거 조작 시간 확보
df['is_monday_claim'] = (df['claim_day_of_week'] == 'Monday').astype(int)

# E4. 요일 인코딩 (ordinal + cyclical)
day_order = {'Monday':0,'Tuesday':1,'Wednesday':2,'Thursday':3,
             'Friday':4,'Saturday':5,'Sunday':6}
df['day_of_week_num'] = df['claim_day_of_week'].map(day_order)
df['day_of_week_sin'] = np.sin(2*np.pi*df['day_of_week_num']/7)
df['day_of_week_cos'] = np.cos(2*np.pi*df['day_of_week_num']/7)

# E5. 월 인코딩 (cyclical)
df['month_sin'] = np.sin(2*np.pi*df['claim_month']/12)
df['month_cos'] = np.cos(2*np.pi*df['claim_month']/12)

# E5. 월말 청구 플래그 (25일 이후)
# 월말·연말은 재정 압박이 커지는 시기 → 사기 동기 증가
df['is_month_end_claim'] = (df['claim_day'] >= 25).astype(int)

# E6. 연말 청구 플래그 (11~12월)
# 연말 지출 증가 → 보험금 필요성 증가
df['is_year_end_claim'] = df['claim_month'].isin([11, 12]).astype(int)

# E7. 겨울철 청구 (12, 1, 2월): 도로 결빙 등 사고 많음
# 여름철 청구 (6, 7, 8월): 장거리 여행·휴가철 = 목격자 부재 가능성
df['is_winter_claim'] = df['claim_month'].isin([12, 1, 2]).astype(int)
df['is_summer_claim'] = df['claim_month'].isin([6, 7, 8]).astype(int)

# E8. 주말+목격자없음 복합 패턴
df['weekend_no_witness'] = (
    (df['is_weekend_claim'] == 1) & (df['no_verification'] == 1)
).astype(int)

In [12]:
temporal_features = [
    'is_weekend_claim',
    'is_monday_claim',
    'day_of_week_sin',
    'day_of_week_cos',
    'month_sin',
    'month_cos',
    'is_month_end_claim',
    'is_year_end_claim',
    'is_winter_claim',
    'is_summer_claim',
    'weekend_no_witness'
]

summary_E = []

for col in temporal_features:
    summary_E.append({
        'feature': col,
        'train_cnt': df.loc[df['_is_train']==1, col].sum() if col in df.columns else None,
        'test_cnt': df.loc[df['_is_train']==0, col].sum() if col in df.columns else None,
        'train_rate': df.loc[df['_is_train']==1, col].mean() if col in df.columns else None,
        'test_rate': df.loc[df['_is_train']==0, col].mean() if col in df.columns else None
    })

summary_E_df = pd.DataFrame(summary_E)

print("\n[E] Temporal Feature Summary")
display(summary_E_df)


[E] Temporal Feature Summary


,feature,train_cnt,test_cnt,train_rate,test_rate
0,is_weekend_claim,5201.000000,3431.000000,0.288944,0.285917
1,is_monday_claim,2578.000000,1738.000000,0.143222,0.144833
2,day_of_week_sin,-44.764632,-11.091879,-0.002487,-0.000924
3,day_of_week_cos,51.453724,17.051310,0.002859,0.001421
4,month_sin,-43.571106,8.935935,-0.002421,0.000745
5,month_cos,7.863448,-163.233938,0.000437,-0.013603
6,is_month_end_claim,3694.000000,2617.000000,0.205222,0.218083
7,is_year_end_claim,3018.000000,1930.000000,0.167667,0.160833
8,is_winter_claim,4521.000000,2885.000000,0.251167,0.240417
9,is_summer_claim,4608.000000,3026.000000,0.256000,0.252167


---
## F. 지역 집계 피처 (Geographic Aggregation Features)

**사기 조사관 관점**: 특정 zip_code에 사기 청구가 집중되는 '핫스팟' 현상이 실제 보험 사기에서 자주 발견

**⚠️ Data Leakage 방지 전략**: Target Encoding은 반드시 Train-only 통계로 계산하고, K-Fold 교차 검증 시 Out-of-Fold(OOF) 방식을 사용

In [13]:
# F1. zip_code 기반 집계 (Train 데이터만으로 계산)
train_only = df[df['_is_train'] == 1].copy()

# zip_code별 청구 건수 (많을수록 핫스팟)
zip_claim_count = train_only.groupby('zip_code').size().reset_index(name='zip_claim_count')

# zip_code별 평균 청구 예상액
zip_avg_payout = (
    train_only.groupby('zip_code')['claim_est_payout']
    .mean()
    .reset_index(name='zip_avg_payout')
)

# zip_code별 평균 과거 청구 수
zip_avg_past_claims = (
    train_only.groupby('zip_code')['past_num_of_claims']
    .mean()
    .reset_index(name='zip_avg_past_claims')
)

In [14]:
# F2. Target Encoding (leakage 방지 OOF 방식)

# 설정값
N_SPLITS = 5
RANDOM_STATE = 2542
SMOOTHING = 10

# train / test 분리
train_idx = df[df['_is_train'] == 1].index
test_idx  = df[df['_is_train'] == 0].index

train_df = df.loc[train_idx].copy()
test_df  = df.loc[test_idx].copy()

# 결과 저장 컬럼 초기화
df['zip_fraud_rate_oof'] = np.nan

# 전체 train 기준 global fraud rate
global_fraud_rate = train_df['fraud'].mean()

# Stratified KFold
skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

# -----------------------------
# OOF Encoding for train
# -----------------------------
for fold, (fit_idx, val_idx) in enumerate(skf.split(train_df, train_df['fraud']), 1):
    fit_fold = train_df.iloc[fit_idx].copy()
    val_fold = train_df.iloc[val_idx].copy()

    # zip_code별 fraud 통계
    zip_stats = fit_fold.groupby('zip_code').agg(
        fraud_sum=('fraud', 'sum'),
        cnt=('fraud', 'count')
    ).reset_index()

    # smoothing 적용
    zip_stats['zip_fraud_rate_oof'] = (
        (zip_stats['fraud_sum'] + global_fraud_rate * SMOOTHING) /
        (zip_stats['cnt'] + SMOOTHING)
    )

    # validation fold에 매핑
    val_encoded = val_fold[['zip_code']].merge(
        zip_stats[['zip_code', 'zip_fraud_rate_oof']],
        on='zip_code',
        how='left'
    )['zip_fraud_rate_oof'].fillna(global_fraud_rate).values

    # 원래 df에 반영
    df.loc[val_fold.index, 'zip_fraud_rate_oof'] = val_encoded

# -----------------------------
# Full-train encoding for test
# -----------------------------
zip_stats_full = train_df.groupby('zip_code').agg(
    fraud_sum=('fraud', 'sum'),
    cnt=('fraud', 'count')
).reset_index()

zip_stats_full['zip_fraud_rate_oof'] = (
    (zip_stats_full['fraud_sum'] + global_fraud_rate * SMOOTHING) /
    (zip_stats_full['cnt'] + SMOOTHING)
)

test_encoded = test_df[['zip_code']].merge(
    zip_stats_full[['zip_code', 'zip_fraud_rate_oof']],
    on='zip_code',
    how='left'
)['zip_fraud_rate_oof'].fillna(global_fraud_rate).values

df.loc[test_df.index, 'zip_fraud_rate_oof'] = test_encoded

# -----------------------------
# High-risk zip flag (OOF 기준)
# -----------------------------
zip_high_risk_threshold_oof = df.loc[df['_is_train'] == 1, 'zip_fraud_rate_oof'].quantile(0.75)

df['is_high_risk_zip_oof'] = (
    df['zip_fraud_rate_oof'] > zip_high_risk_threshold_oof
).astype(int)

# -----------------------------
# 확인
# -----------------------------
print("[K-Fold OOF Target Encoding 완료]")
print(f"Global fraud rate: {global_fraud_rate:.4f}")
print(f"High-risk zip threshold (OOF): {zip_high_risk_threshold_oof:.4f}")

display(
    df.loc[df['_is_train'] == 1, ['zip_code', 'fraud', 'zip_fraud_rate_oof', 'is_high_risk_zip_oof']].head()
)

[K-Fold OOF Target Encoding 완료]
Global fraud rate: 0.1582
High-risk zip threshold (OOF): 0.1843


,zip_code,fraud,zip_fraud_rate_oof,is_high_risk_zip_oof
0,85027,0.0,0.073958,0
1,85004,0.0,0.109704,0
2,85027,0.0,0.102169,0
3,15028,1.0,0.206266,1
4,20148,0.0,0.124713,0


In [15]:
# F3. zip_code 고위험 지역 플래그
zip_high_risk_threshold = df.loc[df['_is_train']==1, 'zip_fraud_rate_oof'].quantile(0.75)
df['is_high_risk_zip'] = (df['zip_fraud_rate_oof'] > zip_high_risk_threshold).astype(int)

In [16]:
geo_features = [
    'zip_claim_count',
    'zip_avg_payout',
    'zip_avg_past_claims',
    'zip_fraud_rate_oof',
    'is_high_risk_zip'
]

summary_F = []

for col in geo_features:
    if col in df.columns:
        summary_F.append({
            'feature': col,
            'train_mean': df.loc[df['_is_train']==1, col].mean(),
            'test_mean': df.loc[df['_is_train']==0, col].mean(),
            'train_std': df.loc[df['_is_train']==1, col].std(),
            'test_std': df.loc[df['_is_train']==0, col].std(),
            'train_null_rate': df.loc[df['_is_train']==1, col].isna().mean(),
            'test_null_rate': df.loc[df['_is_train']==0, col].isna().mean()
        })

summary_F_df = pd.DataFrame(summary_F)

print("\n[F] Geographic Feature Summary")
display(summary_F_df)


[F] Geographic Feature Summary


,feature,train_mean,test_mean,train_std,test_std,train_null_rate,test_null_rate
0,zip_fraud_rate_oof,0.158226,0.158888,0.041454,0.038764,0.0,0.0
1,is_high_risk_zip,0.248056,0.249833,0.431896,0.432934,0.0,0.0


---
## G. 종합 리스크 스코어 (Composite Risk Score)

**사기 조사관 관점**: 개별 위험 신호가 몇 개나 동시에 켜지는가를 집계. 사기 케이스는 보통 3개 이상의 경고 신호가 동시에 발생

In [17]:
# G1. 이진 위험 신호 누적 점수
# 각 항목은 독립적으로 사기를 암시하는 바이너리 신호
risk_flags = [
    'age_is_outlier',           # 비정상 나이
    'low_safety_flag',          # 낮은 안전등급
    'is_low_income',            # 저소득
    'is_repeat_claimer',        # 반복 청구자
    'is_serial_claimer',        # 상습 청구자
    'no_verification',          # 목격자 없음·미확인
    'full_liability_flag',      # 100% 책임 주장
    'no_police_report',         # 경찰 미신고
    'over_value_claim_flag',    # 잔존가치 초과 청구
    'high_payout_ratio_flag',   # 차량 원가의 50% 이상 청구
    'old_vehicle_high_claim',   # 구형차 고액청구
    'overpriced_vehicle_flag',  # 소득 대비 고가 차량
    'no_witness_no_report',     # 목격자없음+경찰미신고
    'unstable_lifestyle',       # 불안정한 생활패턴+반복청구
    'broker_no_report',         # 브로커+경찰미신고
    'is_weekend_claim',         # 주말 청구
    'is_high_risk_zip',         # 고위험 지역
    'address_change_ind',       # 주소 변경
    'accident_parking',         # 주차장 사고
    'extreme_liability_flag',   # 극단적 책임 비율
]

# 존재하는 컬럼만 사용
available_flags = [c for c in risk_flags if c in df.columns]
df['composite_risk_score'] = df[available_flags].sum(axis=1)

# G2. 리스크 스코어 구간별 등급
def risk_grade(score):
    if score <= 2:  return 0  # 저위험
    elif score <= 5:  return 1  # 중위험
    elif score <= 9:  return 2  # 고위험
    else:             return 3  # 매우 고위험

df['risk_grade'] = df['composite_risk_score'].apply(risk_grade)

In [18]:
# G3. 리스크 등급별 사기율 확인 (Train)
train_check = df[df['_is_train'] == 1]
risk_fraud_table = train_check.groupby('risk_grade').agg(
    count=('fraud', 'count'),
    fraud_rate=('fraud', 'mean')
).reset_index()
risk_fraud_table.columns = ['리스크등급', '건수', '사기율']
risk_fraud_table['사기율'] = risk_fraud_table['사기율'].map('{:.2%}'.format)
print("[G] 종합 리스크 스코어 생성 완료")
print(f"  사용된 위험 신호 수: {len(available_flags)}개")
print("\n리스크 등급별 사기율:")
print(risk_fraud_table.to_string(index=False))

[G] 종합 리스크 스코어 생성 완료
  사용된 위험 신호 수: 20개

리스크 등급별 사기율:
 리스크등급   건수    사기율
     0 2527 14.76%
     1 8579 15.69%
     2 6460 16.39%
     3  434 16.13%


---
## H. 최종 피처셋 정리 및 저장

Train / Test 분리 후 사용할 피처 목록 정의

In [19]:
# H1. Train / Test 재분리
train_fe = df[df['_is_train'] == 1].drop(columns=['_is_train']).copy()
test_fe  = df[df['_is_train'] == 0].drop(columns=['_is_train', 'fraud']).copy()

print(f"Train 피처 엔지니어링 완료: {train_fe.shape}")
print(f"Test  피처 엔지니어링 완료: {test_fe.shape}")

Train 피처 엔지니어링 완료: (18000, 103)
Test  피처 엔지니어링 완료: (12000, 102)


In [20]:
# H2. 최종 피처 목록 정의
# ID 컬럼, 원본 날짜 컬럼, 타겟 컬럼 제외
DROP_COLS = [
    'claim_number',          # ID
    'zip_code',              # ID (집계 피처로 대체됨)
    'claim_date',            # 날짜 (분해됨)
    'fraud',                 # 타겟
    'is_high_risk_zip'       # is_high_risk_zip_oof와 중복
]

# 원본 범주형 컬럼 (인코딩 필요하거나 이미 파생 피처로 대체)
ORIGINAL_CATEGORICAL = [
    'gender',               # → 직접 인코딩 또는 유지
    'living_status',        # → living_status_rent로 대체
    'accident_site',        # → accident_* 플래그로 대체
    'channel',              # → channel_* 플래그로 대체
    'vehicle_category',     # → vehicle_size_encoded로 대체
    'vehicle_color',        # 색상은 사기와 직접 연관 낮음 (필요시 유지)
    'claim_day_of_week',    # → day_of_week_num + weekend 플래그로 대체
]

feature_cols = [
    c for c in train_fe.columns
    if c not in DROP_COLS + ORIGINAL_CATEGORICAL
]

In [21]:
print(f"\n최종 피처 수: {len(feature_cols)}개")
print("\n=== 피처 목록 ===")

# 카테고리별 분류 출력
categories = {
    'A. 운전자 리스크':   [c for c in feature_cols if any(k in c for k in ['age','safety','income','driver'])],
    'B. 청구 행동':       [c for c in feature_cols if any(k in c for k in ['claim','witness','liab','report','payout','accident'])],
    'C. 차량 이상':       [c for c in feature_cols if any(k in c for k in ['vehicle','weight','residual','price_per'])],
    'D. 상호작용':        [c for c in feature_cols if any(k in c for k in ['ratio','interaction','overpriced','unstable','broker','repeat','unsafe','young_exp','low_income'])],
    'E. 날짜·시간':       [c for c in feature_cols if any(k in c for k in ['month','year','quarter','day','week','winter','summer'])],
    'F. 지역 집계':       [c for c in feature_cols if 'zip' in c],
    'G. 리스크 스코어':   [c for c in feature_cols if any(k in c for k in ['composite','risk_grade'])],
}

categorized = set()
for cat, cols in categories.items():
    deduped = [c for c in cols if c not in categorized]
    print(f"\n{cat} ({len(deduped)}개):")
    for col in deduped:
        print(f"  - {col}")
    categorized.update(deduped)

uncategorized = [c for c in feature_cols if c not in categorized]
if uncategorized:
    print(f"\n기타 ({len(uncategorized)}개):")
    for col in uncategorized:
        print(f"  - {col}")


최종 피처 수: 92개

=== 피처 목록 ===

A. 운전자 리스크 (16개):
  - age_of_driver
  - annual_income
  - age_of_vehicle
  - age_is_outlier
  - is_young_driver
  - is_senior_driver
  - safety_risk_score
  - low_safety_flag
  - is_low_income
  - is_high_income
  - log_annual_income
  - vehicle_to_income_ratio
  - payout_to_income_ratio
  - low_income_overpriced_vehicle
  - low_income_high_claim
  - income_safety_interaction

B. 청구 행동 (43개):
  - past_num_of_claims
  - witness_present_ind
  - liab_prct
  - policy_report_filed_ind
  - claim_est_payout
  - is_repeat_claimer
  - is_serial_claimer
  - is_extreme_claimer
  - log_past_claims
  - witness_absent
  - witness_unknown
  - full_liability_flag
  - zero_liability_flag
  - extreme_liability_flag
  - no_police_report
  - zero_payout_claim
  - log_claim_payout
  - high_payout_flag
  - extreme_payout_flag
  - accident_parking
  - accident_highway
  - accident_local
  - payout_to_residual_ratio
  - over_value_claim_flag
  - payout_to_price_ratio
  - high_pay

In [22]:
# H3. 결측치 최종 처리
# marital_status: 2개 결측 → 최빈값(1.0) 대체
# marital_status: 그대로 둠 (결측 2건 → 이후 dropna 또는 모델에서 자체 처리)

# witness_present_ind
# → 원본 컬럼은 0(목격자 없음)으로 대체
train_fe['witness_present_ind'] = train_fe['witness_present_ind'].fillna(0).astype(int)
test_fe['witness_present_ind']  = test_fe['witness_present_ind'].fillna(0).astype(int)

# 날짜 파싱 실패로 NaN 된 날짜 파생 피처
date_derived = ['claim_month','claim_year','claim_quarter','claim_day','day_of_week_num']
for col in date_derived:
    if col in train_fe.columns:
        median_val = train_fe[col].median()
        train_fe[col] = train_fe[col].fillna(median_val)
        test_fe[col]  = test_fe[col].fillna(median_val)

# 수치형 나머지 결측 → 0 (이진 플래그들)
train_fe[feature_cols] = train_fe[feature_cols].fillna(0)
test_fe[feature_cols]  = test_fe[feature_cols].fillna(0)

print("[H3] 결측치 처리 완료")
print(f"  Train 잔여 결측: {train_fe[feature_cols].isna().sum().sum()}")
print(f"  Test  잔여 결측: {test_fe[feature_cols].isna().sum().sum()}")

[H3] 결측치 처리 완료
  Train 잔여 결측: 0
  Test  잔여 결측: 0


In [23]:
# H4. 피처 중요도 사전 검토
# 각 이진 피처에서 사기율이 얼마나 다른지 확인

binary_flags = [c for c in feature_cols if train_fe[c].nunique() == 2]
results = []
for col in binary_flags:
    group = train_fe.groupby(col)['fraud'].mean()
    if 0 in group.index and 1 in group.index:
        lift = group[1] / (group[0] + 1e-9)
        results.append({
            '피처': col,
            '플래그=0 사기율': f"{group[0]:.2%}",
            '플래그=1 사기율': f"{group[1]:.2%}",
            'Lift(1/0)': round(lift, 3)
        })

saliency_df = pd.DataFrame(results).sort_values('Lift(1/0)', ascending=False)
print("=== 이진 피처별 사기율 Lift (높을수록 강한 사기 신호) ===")
print(saliency_df.to_string(index=False))

=== 이진 피처별 사기율 Lift (높을수록 강한 사기 신호) ===
                           피처 플래그=0 사기율 플래그=1 사기율  Lift(1/0)
               witness_absent    10.89%    17.40%      1.597
              no_verification    10.90%    17.34%      1.590
             accident_highway    14.33%    21.55%      1.504
           unstable_lifestyle    15.09%    21.56%      1.428
           address_change_ind    12.76%    18.10%      1.419
              low_safety_flag    14.99%    19.27%      1.285
               accident_local    14.14%    17.56%      1.242
      young_expensive_vehicle    15.79%    19.46%      1.232
           living_status_rent    14.67%    17.26%      1.176
            repeat_no_witness    14.96%    17.47%      1.168
      policy_report_filed_ind    14.71%    16.55%      1.125
          extreme_payout_flag    15.74%    17.33%      1.101
         is_high_risk_zip_oof    15.49%    16.84%      1.088
           is_extreme_claimer    15.74%    17.11%      1.087
        unsafe_full_liability    15.81%    17

In [24]:
# H5. 최종 데이터셋 저장
X_train = train_fe[feature_cols]
y_train = train_fe['fraud']
X_test  = test_fe[feature_cols]

print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}  (fraud={y_train.mean():.2%})")
print(f"X_test:  {X_test.shape}")

# CSV 저장
X_train.assign(fraud=y_train.values).to_csv('train_V2.csv', index=False)
X_test.to_csv('test_V2.csv', index=False)

X_train: (18000, 92)
y_train: (18000,)  (fraud=15.82%)
X_test:  (12000, 92)
